In [ ]:
"""
Stack Overflow 2025 Survey - LLM Preference Patterns Analysis
==============================================================
Research Question: Which LLMs do different developer segments prefer, and why?

Analysis Components:
1. Market Basket Analysis - LLM co-usage patterns
2. Correspondence Analysis - LLM-developer profile mapping
3. Gap Analysis - Current vs Desired LLM usage

Author: Analysis Team
Date: 2025
"""

import pandas as pd
import numpy as np
import warnings
from itertools import combinations
from collections import Counter
import os
import sys

# Import matplotlib with error handling
try:
    import matplotlib
    matplotlib.use('Agg')  # Use non-interactive backend
    import matplotlib.pyplot as plt
except Exception as e:
    print(f"Warning: Matplotlib import issue: {e}")
    print("Attempting to reinstall matplotlib...")
    import subprocess
    subprocess.check_call(['pip', 'install', '--upgrade', '--force-reinstall', 'matplotlib'])
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

import seaborn as sns

# Advanced analytics
try:
    from mlxtend.frequent_patterns import apriori, association_rules
except ImportError:
    print("Installing mlxtend...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'mlxtend'])
    from mlxtend.frequent_patterns import apriori, association_rules

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import chi2_contingency, chisquare
from scipy.cluster.hierarchy import dendrogram, linkage

# Network analysis
try:
    import networkx as nx
except ImportError:
    print("Installing networkx...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'networkx'])
    import networkx as nx

# Visualization styling
plt.style.use('default')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def safe_save_csv(df, filepath):
    """Safely save DataFrame to CSV with error handling"""
    try:
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        df.to_csv(filepath, index=False)
        print(f"   ✓ Saved to: {filepath}")
        return True
    except Exception as e:
        print(f"   ⚠ Error saving {filepath}: {str(e)}")
        return False

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_PATH = '../data/processed/cleaned_full.csv'
FIGURES_PATH = '../visualizations/'
TABLES_PATH = '../results/tables/'
MIN_SUPPORT = 0.05  # 5% minimum support for association rules
MIN_CONFIDENCE = 0.3
RANDOM_STATE = 42

# Create all necessary directories
for path in [FIGURES_PATH, TABLES_PATH]:
    os.makedirs(path, exist_ok=True)

# LLM tools to analyze (based on survey schema)
LLM_TOOLS = [
    'ChatGPT', 'Claude', 'GitHub Copilot', 'Gemini', 
    'Perplexity', 'Bing AI', 'Other AI tools'
]

print("="*80)
print("LLM PREFERENCE PATTERNS ANALYSIS")
print("="*80)

# ============================================================================
# STEP 1: LOAD AND PREPARE DATA
# ============================================================================

print("\n[1/7] Loading data...")
try:
    df = pd.read_csv(DATA_PATH, low_memory=False)
    print(f"   ✓ Loaded {len(df):,} responses")
except FileNotFoundError:
    print(f"   ✗ Error: File not found at {DATA_PATH}")
    print("   Please check the file path and try again.")
    sys.exit(1)

# Identify LLM-related columns
llm_columns = [col for col in df.columns if col in ['AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith', 'AIModelsAdmired']]

# If those don't exist, try to find any LLM-related columns
if len(llm_columns) == 0:
    llm_columns = [col for col in df.columns if 'AIModels' in col]

print(f"   ✓ Found {len(llm_columns)} LLM usage columns")
print(f"   LLM columns: {llm_columns}")

if len(llm_columns) == 0:
    print("   ✗ Error: No LLM columns found in dataset")
    print("   Available AI-related columns:")
    for col in [c for c in df.columns if 'AI' in c][:15]:
        print(f"     - {col}")
    sys.exit(1)

# Create subset for LLM analysis
required_columns = ['DevType', 'YearsCode', 'Country', 'experience_category']
available_columns = [col for col in required_columns if col in df.columns]
df_llm = df[llm_columns + available_columns].copy()

# Remove rows with no LLM usage data
df_llm = df_llm.dropna(subset=llm_columns, how='all')
print(f"   ✓ Analyzing {len(df_llm):,} respondents with LLM data")

# ============================================================================
# STEP 2: BASIC LLM USAGE STATISTICS
# ============================================================================

print("\n[2/7] Computing basic usage statistics...")

# Parse semicolon-separated values to extract individual tools
if 'AIModelsHaveWorkedWith' in llm_columns:
    # Extract unique LLM names from the text column
    all_llms = []
    for val in df['AIModelsHaveWorkedWith'].dropna():
        if isinstance(val, str):
            tools = [t.strip() for t in val.split(';')]
            all_llms.extend(tools)
    
    llm_adoption = {}
    for tool in set(all_llms):
        count = df['AIModelsHaveWorkedWith'].astype(str).str.contains(tool, case=False, na=False).sum()
        adoption_rate = (count / len(df)) * 100
        llm_adoption[tool] = adoption_rate
    
    llm_adoption_df = pd.DataFrame.from_dict(
        llm_adoption, orient='index', columns=['Adoption Rate (%)']
    ).sort_values('Adoption Rate (%)', ascending=False)
    
    print("\n   LLM Adoption Rates (Top 10):")
    for tool, rate in llm_adoption_df.head(10).iterrows():
        print(f"     • {tool}: {rate['Adoption Rate (%)']:.1f}%")
    
    # Visualize adoption rates
    try:
        fig, ax = plt.subplots(figsize=(10, 6))
        llm_adoption_df.head(10).plot(kind='barh', ax=ax, legend=False, color='steelblue')
        ax.set_xlabel('Adoption Rate (%)')
        ax.set_title('Top 10 LLM Tool Adoption Rates Among Developers', fontsize=14, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{FIGURES_PATH}llm_adoption_rates.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("   ✓ Saved: llm_adoption_rates.png")
        
        # Save adoption data
        safe_save_csv(llm_adoption_df.reset_index(), f'{TABLES_PATH}llm_adoption_rates.csv')
    except Exception as e:
        print(f"   ⚠ Could not save plot: {e}")
else:
    print("   ⚠ AIModelsHaveWorkedWith column not found")
    llm_adoption_df = pd.DataFrame()

# ============================================================================
# STEP 3: MARKET BASKET ANALYSIS - LLM CO-USAGE PATTERNS
# ============================================================================

print("\n[3/7] Running Market Basket Analysis...")

# Build transaction data from semicolon-separated values
transactions = []
for val in df['AIModelsHaveWorkedWith'].dropna():
    if isinstance(val, str):
        tools = set([t.strip() for t in val.split(';')])
        transactions.append(tools)

if transactions:
    # Create binary matrix
    all_tools = set()
    for trans in transactions:
        all_tools.update(trans)
    
    all_tools = sorted(list(all_tools))
    basket_data = []
    for trans in transactions:
        row = [tool in trans for tool in all_tools]
        basket_data.append(row)
    
    basket_df = pd.DataFrame(basket_data, columns=all_tools)
    
    # Find frequent itemsets using Apriori
    try:
        frequent_itemsets = apriori(basket_df, min_support=MIN_SUPPORT, use_colnames=True)
        print(f"   ✓ Found {len(frequent_itemsets)} frequent itemsets")
        
        # Generate association rules
        if len(frequent_itemsets) > 1:
            rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
            rules = rules.sort_values('lift', ascending=False)
            
            print(f"   ✓ Generated {len(rules)} association rules")
            print("\n   Top 5 LLM Combinations (by Lift):")
            
            for idx, row in rules.head(5).iterrows():
                antecedent = ', '.join(list(row['antecedents']))
                consequent = ', '.join(list(row['consequents']))
                print(f"     {antecedent} → {consequent}")
                print(f"       Support: {row['support']:.3f} | Confidence: {row['confidence']:.3f} | Lift: {row['lift']:.3f}")
            
            # Save association rules
            safe_save_csv(rules, f'{TABLES_PATH}llm_association_rules.csv')
        else:
            print("   ⚠ Not enough frequent itemsets for association rules")
            rules = pd.DataFrame()
    except Exception as e:
        print(f"   ⚠ Market basket analysis failed: {e}")
        frequent_itemsets = pd.DataFrame()
        rules = pd.DataFrame()
else:
    print("   ⚠ No transaction data available")
    frequent_itemsets = pd.DataFrame()
    rules = pd.DataFrame()

# ============================================================================
# STEP 4: NETWORK ANALYSIS - SKIPPED
# ============================================================================

print("\n[4/7] Network analysis skipped...")
print("   ⚠ Network visualization disabled")

# Initialize empty variables for later use
centrality = {}

# ============================================================================
# STEP 5: CORRESPONDENCE ANALYSIS - LLM × DEVELOPER PROFILES
# ============================================================================

print("\n[5/7] Running Correspondence Analysis...")

if 'DevType' in df.columns:
    # Create contingency table: DevType × LLM tools
    dev_types = df['DevType'].dropna().value_counts().head(10).index.tolist()
    
    # Build contingency table
    contingency_data = []
    for dev_type in dev_types:
        subset = df[df['DevType'] == dev_type]
        row = []
        for tool in (all_tools if 'all_tools' in locals() else []):
            count = subset['AIModelsHaveWorkedWith'].astype(str).str.contains(tool, case=False, na=False).sum()
            row.append(count)
        contingency_data.append(row)
    
    contingency_table = pd.DataFrame(
        contingency_data, 
        index=dev_types,
        columns=(all_tools if 'all_tools' in locals() else [])
    )
    
    print(f"   ✓ Built contingency table: {contingency_table.shape}")
    
    # Chi-square test for independence - with guard for empty tables
    if contingency_table.size > 0 and contingency_table.values.sum() > 0:
        try:
            chi2, p_value, dof, expected = chi2_contingency(contingency_table)
            total = contingency_table.sum().sum()
            min_dim = min(contingency_table.shape[0]-1, contingency_table.shape[1]-1)
            
            if min_dim > 0 and total > 0:
                cramers_v = np.sqrt(chi2 / (total * min_dim))
            else:
                cramers_v = 0
            
            print(f"\n   Chi-square test for independence:")
            print(f"     χ² = {chi2:.2f}, p-value = {p_value:.4e}")
            print(f"     Cramér's V (effect size) = {cramers_v:.3f}")
            
            if p_value < 0.05:
                print("     ✓ Significant relationship between DevType and LLM preference")
            else:
                print("     ✗ No significant relationship detected")
        except Exception as e:
            print(f"   ⚠ Chi-square test failed: {e}")
    else:
        print("   ⚠ Contingency table is empty or insufficient data")
    
    # Visualize with heatmap
    if contingency_table.size > 0:
        try:
            fig, ax = plt.subplots(figsize=(12, 8))
            sns.heatmap(contingency_table.T, annot=True, fmt='g', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Usage Count'})
            ax.set_title('LLM Preferences by Developer Type\n(Heatmap: Higher values = more usage)', 
                         fontsize=14, fontweight='bold')
            ax.set_xlabel('Developer Type')
            ax.set_ylabel('LLM Tool')
            plt.tight_layout()
            plt.savefig(f'{FIGURES_PATH}llm_devtype_heatmap.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("   ✓ Saved: llm_devtype_heatmap.png")
            
            # Save contingency table
            safe_save_csv(contingency_table.reset_index(), f'{TABLES_PATH}llm_devtype_contingency.csv')
        except Exception as e:
            print(f"   ⚠ Could not save heatmap: {e}")
else:
    print("   ⚠ DevType column not found in dataset")

# ============================================================================
# STEP 6: ADMIRED VS DESIRED GAP ANALYSIS
# ============================================================================

print("\n[6/7] Analyzing Admired vs Desired gap...")

if 'AIModelsHaveWorkedWith' in df.columns and 'AIModelsWantToWorkWith' in df.columns:
    try:
        # Extract unique tools from both columns
        current_tools = Counter()
        desired_tools = Counter()
        
        for val in df['AIModelsHaveWorkedWith'].dropna():
            if isinstance(val, str):
                for tool in val.split(';'):
                    current_tools[tool.strip()] += 1
        
        for val in df['AIModelsWantToWorkWith'].dropna():
            if isinstance(val, str):
                for tool in val.split(';'):
                    desired_tools[tool.strip()] += 1
        
        # Calculate gap
        all_gap_tools = set(current_tools.keys()) | set(desired_tools.keys())
        gap_df = pd.DataFrame({
            'Current': pd.Series({tool: current_tools.get(tool, 0) for tool in all_gap_tools}),
            'Desired': pd.Series({tool: desired_tools.get(tool, 0) for tool in all_gap_tools})
        })
        gap_df['Gap'] = gap_df['Desired'] - gap_df['Current']
        gap_df = gap_df.sort_values('Gap', ascending=False)
        
        print("\n   Top Tools by Desire Gap:")
        for tool, row in gap_df.head(10).iterrows():
            if row['Gap'] > 0:
                print(f"     • {tool}: Current={row['Current']:.0f}, Desired={row['Desired']:.0f}, Gap={row['Gap']:.0f}")
        
        # Visualize gap
        try:
            fig, ax = plt.subplots(figsize=(10, 6))
            gap_data = gap_df.head(10)[['Current', 'Desired']].fillna(0)
            gap_data.plot(kind='barh', ax=ax)
            ax.set_title('LLM Tools: Current vs Desired Usage\n(Gap indicates unmet demand)', 
                         fontsize=14, fontweight='bold')
            ax.set_xlabel('Number of Developers')
            ax.legend(['Currently Use', 'Want to Use'])
            ax.grid(axis='x', alpha=0.3)
            plt.tight_layout()
            plt.savefig(f'{FIGURES_PATH}llm_desire_gap.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("   ✓ Saved: llm_desire_gap.png")
            
            # Save gap analysis
            safe_save_csv(gap_df.reset_index(), f'{TABLES_PATH}llm_gap_analysis.csv')
        except Exception as e:
            print(f"   ⚠ Could not save gap analysis: {e}")
    except Exception as e:
        print(f"   ⚠ Gap analysis failed: {e}")
else:
    print("   ⚠ Required columns not available in dataset")

# ============================================================================
# STEP 7: SUMMARY & KEY INSIGHTS
# ============================================================================

print("\n[7/7] Generating summary report...")
print("\n" + "="*80)
print("ANALYSIS COMPLETE - KEY INSIGHTS")
print("="*80)

print("\n📊 Market Basket Insights:")
if len(frequent_itemsets) > 0:
    print(f"   • {len(rules)} significant LLM co-usage patterns identified")
    if len(rules) > 0:
        print(f"   • Strongest association: Lift = {rules['lift'].max():.2f}")
else:
    print("   • Insufficient data for market basket analysis")

print("\n📈 Recommendations:")
print("   1. High co-usage pairs indicate complementary tool ecosystems")
print("   2. Tools with high desire-gap represent growth opportunities")
print("   3. Developer type preferences show segment-specific adoption patterns")

print("\n✅ All visualizations saved to:", FIGURES_PATH)
print("✅ All tables saved to:", TABLES_PATH)
print("="*80)

LLM PREFERENCE PATTERNS ANALYSIS

[1/7] Loading data...
   ✓ Loaded 49,123 responses
   ✓ Found 3 LLM usage columns
   LLM columns: ['AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith', 'AIModelsAdmired']
   ✓ Analyzing 16,261 respondents with LLM data

[2/7] Computing basic usage statistics...

   LLM Adoption Rates (Top 10):
     • Anthropic: Claude Sonnet: 14.4%
     • openAI Reasoning models: 11.6%
     • openAI Image generating models: 8.9%
     • X Grok models: 3.7%
     • Mistral AI models: 3.5%
     • Perplexity Sonar models: 2.5%
     • Alibaba Cloud Qwen models: 1.8%
     • Microsoft Phi-4 models: 1.7%
     • Amazon Titan models: 0.6%
     • Cohere: Command A: 0.3%
   ✓ Saved: llm_adoption_rates.png
   ✓ Saved to: ./tables/llm_adoption_rates.csv

[3/7] Running Market Basket Analysis...
   ✓ Found 189 frequent itemsets
   ✓ Generated 1520 association rules

   Top 5 LLM Combinations (by Lift):
     DeepSeek (R- Reasoning models), Gemini (Flash general purpose models) → DeepSeek